# Fraud Modeling: Fraud_Data

This notebook trains and compares two classifiers on the processed e-commerce fraud dataset: a baseline **Logistic Regression** model and a tuned **Random Forest** ensemble. We use a stratified train/test split, apply **SMOTE only to the training set** to address class imbalance, and evaluate on an untouched holdout that reflects real-world fraud prevalence (~9%).

**Goals**
- Load the model-ready feature matrix from `data/processed/fraud_data_features.csv`
- Split data with stratification so both sets retain fraud cases
- Train a simple baseline and a tuned ensemble on the same split
- Evaluate with metrics suited to imbalanced fraud detection
- Compare models side by side and identify the current best performer for interim submission

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from IPython.display import display, Markdown
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GridSearchCV

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RANDOM_STATE, TEST_SIZE
from src.modeling import (
    compare_class_distributions,
    compare_model_results,
    confusion_matrix_frame,
    describe_feature,
    extract_feature_importance,
    format_comparison_table,
    get_coefficient_extremes,
    identify_best_model,
    load_fraud_feature_matrix,
    plot_top_features,
    run_fraud_modeling_workflow,
    save_modeling_report,
    stratified_train_test_split,
    train_classifier,
)
from src.preprocessing import class_imbalance_summary

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Load Processed Features

We use the engineered feature matrix produced by the feature pipeline: numeric features scaled, categoricals one-hot encoded, and the fraud label (`class`) separated from predictors.

In [ ]:
features, target = load_fraud_feature_matrix()

print(f"Feature matrix shape: {features.shape}")
print(f"Target shape: {target.shape}")
print(f"Fraud rate: {target.mean():.2%}")

class_imbalance_summary(target.to_frame(name="class"), target_column="class")

## 2. Stratified Train/Test Split

We hold out **20%** of transactions for evaluation (`TEST_SIZE = 0.2`, `RANDOM_STATE = 42`). Stratification keeps the fraud rate similar in both splits so the test set is representative of production traffic.

In [ ]:
split = stratified_train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(split.x_train):,}")
print(f"Test rows:     {len(split.x_test):,}")
print(f"Features:      {len(split.feature_names)}")

train_dist = class_imbalance_summary(
    split.y_train.to_frame(name="class"), target_column="class"
)
test_dist = class_imbalance_summary(
    split.y_test.to_frame(name="class"), target_column="class"
)

display(
    pd.concat(
        [
            train_dist.assign(split="train"),
            test_dist.assign(split="test"),
        ],
        ignore_index=True,
    )
)

## 3. Handle Class Imbalance on Training Data Only

Fraud is the minority class (~9% of transactions). We apply **SMOTE** to the training set to synthesize additional fraud examples. The test set is **never** resampled — evaluation must reflect the natural class mix customers actually experience.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
x_train_resampled, y_train_resampled = smote.fit_resample(split.x_train, split.y_train)

x_train_resampled = pd.DataFrame(x_train_resampled, columns=split.feature_names)
y_train_resampled = pd.Series(y_train_resampled, name="class")

print(f"Training rows before SMOTE: {len(split.x_train):,}")
print(f"Training rows after SMOTE:  {len(x_train_resampled):,}")

distribution_comparison = compare_class_distributions(
    split.y_train,
    y_train_resampled,
    split.y_test,
)
display(distribution_comparison[["stage", "class", "count", "pct"]])

## 4. Train and Tune All Models

The finalized workflow tunes **Logistic Regression** (`GridSearchCV`), **Random Forest**, and **XGBoost** (`RandomizedSearchCV`) on the pre-SMOTE training split (CV scored on PR-AUC), refits on SMOTE-balanced data, evaluates on the holdout set, and saves report-ready outputs to `reports/outputs/`.

In [ ]:
workflow = run_fraud_modeling_workflow(
    save_report=True,
    store_training_data=True,
)

model_results = workflow.results
comparison = workflow.comparison
best_result = workflow.best_result
best_name = best_result.model_name
best_metrics = best_result.metrics
report_paths = workflow.report_paths

baseline_result = next(r for r in model_results if r.model_name == "logistic_regression")
ensemble_result = next(r for r in model_results if r.model_name == "random_forest")
xgboost_result = next(r for r in model_results if r.model_name == "xgboost")

print(f"Best model (PR-AUC): {best_name}")
display(format_comparison_table(comparison))

### Tuning summary

Each model's cross-validated PR-AUC and best hyperparameters are logged below. Tree-based models use `RandomizedSearchCV` to keep runtime manageable on the full feature matrix.

In [ ]:
tuning_summary = pd.DataFrame(
    [
        {
            "model_name": t.model_name,
            "search_strategy": t.search_strategy,
            "best_cv_pr_auc": t.best_cv_score,
            "best_params": t.best_params,
        }
        for t in workflow.tuning_results
    ]
)
display(tuning_summary)

## 6. Model Comparison

Both models are evaluated on the **same untouched test set** using identical metrics. For fraud detection with ~9% prevalence, **AUC-PR** is our primary ranking metric because it measures how well each model separates fraud from legitimate transactions when the positive class is rare. F1, recall, and precision at the default 0.5 threshold are reported for operational context.

| Metric | Business meaning |
|--------|------------------|
| **Precision** | Of flagged transactions, how many are actually fraud? (controls false alarms) |
| **Recall** | Of all fraud, how much do we catch? (controls missed fraud) |
| **F1** | Balance between precision and recall at the default threshold |
| **ROC-AUC** | Overall ranking ability across thresholds |
| **AUC-PR** | Ranking quality focused on the rare fraud class — primary metric for model selection here |

In [ ]:
comparison_display = format_comparison_table(comparison).copy()

for col in ["precision", "recall", "f1", "roc_auc", "pr_auc"]:
    comparison_display[col] = comparison_display[col].map(lambda v: f"{v:.4f}")

print("Side-by-side holdout metrics (sorted by PR-AUC):")
display(comparison_display)

print("\nReport-ready outputs:")
for name, path in report_paths.as_dict().items():
    print(f"  {name}: {path}")

best_row = comparison.iloc[0]
runner_up_row = comparison.iloc[1]

pr_auc_gap = best_row["pr_auc"] - runner_up_row["pr_auc"]
f1_gap = best_row["f1"] - runner_up_row["f1"]

display(
    Markdown(
        f"**Current best model: `{best_name}`**\n\n"
        f"- **PR-AUC:** {best_metrics.auc_pr:.4f} "
        f"(+{pr_auc_gap:.4f} vs {runner_up_row['model_name']})\n"
        f"- **F1:** {best_metrics.f1:.4f} "
        f"(+{f1_gap:.4f} vs {runner_up_row['model_name']})\n"
        f"- **Recall:** {best_metrics.recall:.4f} | "
        f"**Precision:** {best_metrics.precision:.4f}\n\n"
        f"**Why it leads:** PR-AUC is the primary metric for imbalanced fraud detection "
        f"because it rewards models that rank fraud cases near the top of the alert queue. "
        f"{'Tree-based models often win by learning non-linear feature interactions that a linear baseline cannot represent.'
        if best_name in {'random_forest', 'xgboost'}
        else 'Logistic Regression remains competitive, suggesting much of the signal is linear on this holdout.'}"
    )
)

cm = confusion_matrix_frame(best_result.y_true, best_result.y_pred)
print(f"\nConfusion matrix — {best_name} (rows = actual, columns = predicted):")
display(cm)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Legitimate (0)", "Fraud (1)"],
    yticklabels=["Legitimate (0)", "Fraud (1)"],
    ax=ax,
)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

total_fraud = best_metrics.true_positives + best_metrics.false_negatives
total_legit = best_metrics.true_negatives + best_metrics.false_positives
print(
    f"Fraud caught: {best_metrics.true_positives:,} / {total_fraud:,} "
    f"({best_metrics.true_positives / total_fraud:.1%} recall)"
)
print(
    f"False alarms: {best_metrics.false_positives:,} / {total_legit:,} legitimate transactions "
    f"({best_metrics.false_positives / total_legit:.2%} of legit flagged)"
)

## 7. Precision-Recall and ROC Curves

Overlaying all three models makes the ranking difference visible. Saved report figures: `plots/model_comparison_pr_curve.png` and `plots/model_comparison_roc_curve.png`.

In [ ]:
baseline_prevalence = baseline_result.y_true.mean()

fig, ax = plt.subplots(figsize=(8, 6))

for model_result, color in [
    (baseline_result, "C0"),
    (ensemble_result, "C1"),
    (xgboost_result, "C2"),
]:
    precision_vals, recall_vals, _ = precision_recall_curve(
        model_result.y_true,
        model_result.y_score,
    )
    ax.plot(
        recall_vals,
        precision_vals,
        linewidth=2,
        color=color,
        label=f"{model_result.model_name} (PR-AUC = {model_result.metrics.auc_pr:.3f})",
    )

ax.axhline(
    baseline_prevalence,
    linestyle="--",
    color="gray",
    label=f"No-skill baseline ({baseline_prevalence:.1%} fraud rate)",
)
ax.set_xlabel("Recall (fraud caught)")
ax.set_ylabel("Precision (flags that are fraud)")
ax.set_title("Precision-Recall Curves — Model Comparison")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 8. Feature Importance — Best Model

Before full SHAP analysis, we inspect what the **current best model** relies on most. Logistic Regression exposes signed **coefficients** (positive = higher fraud score); tree ensembles expose **feature importances** (magnitude only). This is a directional sanity check — not a causal proof of fraud drivers.

In [ ]:
importance_table = extract_feature_importance(best_result)

print(f"Report table: {report_paths.top_features_csv}")
print(f"Report figure: {report_paths.feature_importance_plot}")
print(f"Importance type: {importance_table['importance_type'].iloc[0]}")
display(importance_table.head(10))

if importance_table["importance_type"].iloc[0] == "coefficient":
    top_positive, top_negative = get_coefficient_extremes(importance_table, top_n=10)
    print("\nStrongest positive coefficients (increase fraud score):")
    display(top_positive[["feature", "coefficient"]])
    print("\nStrongest negative coefficients (decrease fraud score):")
    display(top_negative[["feature", "coefficient"]])

print("\nTop drivers in plain language:")
for _, row in importance_table.head(5).iterrows():
    print(f"- {row['feature']}: {describe_feature(row['feature'])}")

fig, ax = plt.subplots(figsize=(10, 6))
plot_top_features(importance_table, model_name=best_name, top_n=10, ax=ax)
plt.show()

### What the top features suggest

The ranked features above describe **what the model uses to score transactions**, not guaranteed root causes. Still, they offer useful interim signals for fraud analysts:

- **Transaction value and timing** (`purchase_value`, `hour_of_day`, `time_since_signup_hours`) often appear when fraudsters behave differently from typical shoppers — for example, unusually large purchases or buying soon after account creation.
- **Channel and device context** (`source_*`, `browser_*`) can flag traffic patterns that differ from normal e-commerce journeys, such as certain acquisition channels or browser combinations that co-occur with fraud in this dataset.
- **Geography** (`country_*`) highlights locations where flagged transactions cluster. This is a **risk signal for review**, not a reason to block countries outright — legitimate customers share the same labels.
- **Velocity features** (`txn_count_last_*`, `user_txn_velocity_per_day`) would matter more in datasets with repeat purchasers; in this snapshot most users have a single transaction, so these may rank lower.

**Interim read:** If purchase value, signup timing, or specific channels/countries dominate the top ranks, the business should prioritize rules and review queues around those patterns while planning SHAP-based case explanations next. If importances are spread thinly across many country dummies, the model may be leaning on sparse geographic splits — a flag to validate with deeper explainability before deployment.

## 9. Interim Submission Summary

### Report-ready outputs

Section 6 writes reusable artifacts to `reports/outputs/`:

| File | Use in reports |
|------|----------------|
| `model_comparison_metrics.csv` | Side-by-side model metrics table |
| `best_model_summary.md` | Narrative best-model section (copy into interim/final report) |
| `best_model_summary.csv` | One-row summary for tables or dashboards |
| `top_features_best_model.csv` | Feature importance appendix |
| `plots/model_comparison_pr_curve.png` | Model comparison figure |
| `plots/model_comparison_roc_curve.png` | ROC curve comparison |
| `confusion_matrices/confusion_matrix_*.png` | Per-model confusion matrices |
| `plots/best_model_feature_importance.png` | Top-feature drivers figure |

Regenerate everything from the notebook (section 6) or run:

```bash
python scripts/run_modeling_reports.py
```

### Current best model

The comparison table in section 6 ranks models by **AUC-PR**. The leading model on the stratified holdout is identified dynamically there, along with metric gaps vs the runner-up. Feature importance in section 8 reflects that same best model.

### Why AUC-PR drives the recommendation

Accuracy is high for both models (~90%+) because most transactions are legitimate — it is not a useful differentiator here. **AUC-PR** directly measures ranking quality for the fraud class, which aligns with how fraud teams work: they review the highest-scored transactions first. A higher AUC-PR means more fraud appears near the top of the ranked list, regardless of the exact alert threshold chosen later.

### Honest limitations at this stage

- Both models use a **default 0.5 probability threshold**, which is rarely optimal for fraud operations. Precision and recall will shift once a business-specific threshold is chosen.
- Hyperparameter tuning was **lightweight** (small grid, 3-fold CV on pre-SMOTE data) to keep notebook runtime reasonable — further tuning could change the ranking.
- Feature importance here is **model-native** (coefficients or impurity-based importances), not SHAP — directionality for tree models and interaction effects still need full explainability analysis.
- Results apply to **Fraud_Data only**; the credit card dataset has a very different imbalance profile and will need its own modeling pass.

### Interim takeaway

We now have evidence that engineered features carry predictive signal, a **three-model comparison** (Logistic Regression, Random Forest, XGBoost), and **report-ready CSV/Markdown/plot outputs** under `reports/outputs/`. The best current model is selected automatically by PR-AUC and should be carried forward for threshold tuning and SHAP-based interpretation — but it should not yet be treated as production-ready without those follow-up steps.